### Load the Config, Model ###

In [ ]:
import pandas as pd
import numpy as np
import huggingface 
from time_moe.datasets.time_moe_dataset import TimeMoEDataset
from time_moe.datasets.time_moe_window_dataset import TimeMoEWindowDataset
import datetime
import os
import torch
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    

# Configure logging
logging.basicConfig(    
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

print(f"CWD: {os.getcwd()}")


In [ ]:
# Loading of config
from utils.utils import load_config, config_to_args
from time_moe.models.modeling_time_moe import TimeMoeForPrediction

#config_file_path = "../config/config_tickers_75_7_Channels_with_temporal_tape.yaml"
config_file_path = '../config/config_tickers_448_7_Channels_with_temporal_tape_v2.yaml'
config_dict = load_config(config_file_path)

place_holders_to_replace = {'version_name': config_dict['regression']['common']['version_name']}
args = config_to_args(config_dict, place_holders_to_replace)

print(f"Current Version Name: {args.regression.common.version_name}")


# Loading of the model
device = 'cpu'
logging.info(f"Loading Model from HuggingFace @ {args.regression.Time_MOE.model_path}")
model = TimeMoeForPrediction.from_pretrained(
    args.regression.Time_MOE.model_path, 
    device_map='cpu' if not torch.cuda.is_available() else ('auto' if device is None else device),
    torch_dtype='auto',
)
        

### Given Feature Configs, mutate and generate lots of different sequences, prepping various tensor variants ###

In [ ]:
from time_moe.datasets.time_moe_dataset import MultiFrequencyDatasetLoaderV2

feature_engineered_df = pd.read_parquet(args.regression.S2.feature_engineered_file_path)

dataloader = MultiFrequencyDatasetLoaderV2(
    args,
    feature_engineered_df=feature_engineered_df,
    stage='full',
    batch_size=args.regression.Time_MOE.inference_model_args.batch_size,
    context_length=args.regression.Time_MOE.train_model_args.max_length,
    prediction_length=args.regression.Time_MOE.inference_model_args.prediction_length,
    
    num_workers=3
)

### Initiate Inferencing on these sequences and store them ###

In [ ]:
import torch
import pickle
from tqdm import tqdm

all_sequences = []
with torch.no_grad():
    for batch in tqdm(dataloader):
        #print(f'Input Tensor Shape: {batch['input_ids'].shape}')
        #print(f'Input IDs: {batch['input_ids']}')
        
        # Check for other channels
        # for key in batch:
        #     if key.startswith('channel_'):
        #         #print(f'Channel {key} Shape: {batch[key].shape}')
    
        # Check for NaNs
        if torch.isnan(batch['input_ids']).any():
            print("⚠️ NaN detected in input subsequence_norm!")
            
        # Enter the channels as kwargs
        output = model(
            input_ids=batch['input_ids'],
            max_horizon_length=args.regression.Time_MOE.inference_model_args.prediction_length,
            return_dict=True,
            **{key: batch[key] for key in batch if key.startswith('channel_')}
        )
        
        #print(f'Output Shape: {output.logits.shape}')
        forecast = output.logits[:, -1, :]   # [B, prediction_len, F] simplified

        #print(f'Forecast Shape: {forecast.shape}')
        
        # --- Inverse scaling per subsequence ---
        forecast_original_scale = []
        for i in range(forecast.shape[0]):
            f = forecast[i]                                # [prediction_len, F]
            mean = batch['mean'][i]           # [1, F]
            std = batch['std'][i]             # [1, F]
            f_inv = dataloader.dataset.inverse_scale(f, mean, std)
            forecast_original_scale.append(f_inv)

        forecast_original_scale = torch.stack(forecast_original_scale, dim=0)  # [B, prediction_len, F]

        # Store in batch
        batch['model_prediction_sequence'] = forecast_original_scale
        all_sequences.append(batch)
        
logging.info(f"Model Inferencing Completed!")
 
# Save the sequences 
seq_path = args.regression.S6.output_seq_path
logging.info(f'Saving the individual augmented inferenced sequences')

# Saving the all_sequences as a whole
with open(seq_path, "wb") as f:
    pickle.dump(all_sequences, f)
logging.info(f'Saved the individual augmented inferenced sequences to {seq_path}')